In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.models.falcon_mamba.modeling_falcon_mamba import FalconMambaMixer
from transformers.models.falcon_mamba.modeling_falcon_mamba import rms_forward

def load_model(model_name='tiiuae/falcon-mamba-7b', quantize=False):
    """
    Load the model and tokenizer.
    
    Args:
        model_name (str): Name or path of the model to load
        quantize (bool): Whether to apply 4-bit quantization
        
    Returns:
        tuple: (model, tokenizer)
    """
    
    if quantize:
        quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
        model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quantization_config)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name)
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model.to("cuda:1")
    model.eval()
    return model, tokenizer


# MambaMixer that can be modified

def make_slow_forward_with_hook(orig_slow_forward):
    """
    Returns a new slow_forward method that does exactly what the original
    slow_forward does, except it also saves the intermediate 'scan_output'
    (before multiplying by the gate) in a module attribute, which we can then read later.
    `orig_slow_forward` is the original method we want to wrap.
    """

    # Define an inner function that implements the patched logic.
    def slow_forward_hooked(self, input_states, cache_params=None, cache_position=None, attention_mask=None):
        """
        This function mirrors FalconMambaMixer.slow_forward but includes a step
        to capture the 'scan_output' before gate multiplication.
        """

        # --- replicate the entire original slow_forward logic ---

        # Unpack shapes and data type for convenience
        batch_size, seq_len, _ = input_states.shape
        dtype = input_states.dtype

        # 1. Gated MLP's linear projection: 
        #    We do in_proj -> shape is [batch_size, 2*intermediate_size, seq_len]
        projected_states = self.in_proj(input_states).transpose(1, 2)
        
        # Split into hidden_states and gate. Each has shape [batch_size, intermediate_size, seq_len].
        hidden_states, gate = projected_states.chunk(2, dim=1)

        # If we have an attention mask, apply it to hidden_states.
        if attention_mask is not None:
            # [batch_size, seq_len] -> broadcast along the 'intermediate_size' dimension
            hidden_states = hidden_states * attention_mask.unsqueeze(1)

        # 2. Convolution sequence transformation
        if cache_params is not None:
            # If we have caching, we retrieve the stored SSM state for this layer.
            ssm_state = cache_params.ssm_states[self.layer_idx].clone()
            ssm_state = ssm_state.to(hidden_states.device)

            # cache_position will indicate if we are in a prefill stage or generation stage.
            if cache_position is not None and cache_position.shape[0] == self.conv_kernel_size:
                # This means we are in the very beginning; we pad the hidden_states
                conv_state = torch.nn.functional.pad(
                    hidden_states,
                    (self.conv_kernel_size - hidden_states.shape[-1], 0)
                )
                # Store the padded state in cache
                cache_params.update_conv_state(self.layer_idx, conv_state, cache_position)

                # Perform the convolution, then apply the activation
                hidden_states = self.act(
                    self.conv1d(hidden_states)[..., :seq_len]
                )
            else:
                # If we are in "decoding" step by step, do a specialized update
                conv_state = cache_params.update_conv_state(self.layer_idx, hidden_states, cache_position)
                # Multiply by weight and sum across the sequence dimension
                hidden_states = torch.sum(conv_state * self.conv1d.weight[:, 0, :], dim=-1)
                if self.use_conv_bias:
                    hidden_states += self.conv1d.bias

                # Apply activation (and restore dtype), then unsqueeze for shape consistency
                hidden_states = self.act(hidden_states).to(dtype).unsqueeze(-1)
        else:
            # If we have no caching, just do the usual convolution over the entire sequence
            ssm_state = torch.zeros(
                (batch_size, self.intermediate_size, self.ssm_state_size),
                device=hidden_states.device,
                dtype=dtype
            )
            hidden_states = self.act(
                self.conv1d(hidden_states)[..., :seq_len]
            )

        # Apply attention mask again, if present.
        if attention_mask is not None:
            hidden_states = hidden_states * attention_mask.unsqueeze(1)

        # 3. State Space Model (SSM) sequence transformation
        # 3.a. We compute the input-dependent parameters by projecting hidden_states.
        ssm_parameters = self.x_proj(hidden_states.transpose(1, 2))
        time_step, B, C = torch.split(
            ssm_parameters, [self.time_step_rank, self.ssm_state_size, self.ssm_state_size], dim=-1
        )

        # A small RMS forward pass (normalization) on B, C, and time_step
        B = rms_forward(B, variance_epsilon=self.rms_eps)
        C = rms_forward(C, variance_epsilon=self.rms_eps)
        time_step = rms_forward(time_step, variance_epsilon=self.rms_eps)

        # Discretize the time step: a linear + softplus
        discrete_time_step = self.dt_proj(time_step)
        discrete_time_step = torch.nn.functional.softplus(discrete_time_step).transpose(1, 2)
        # shape -> [batch_size, intermediate_size, seq_len]
        self.delta = discrete_time_step.detach().cpu()

        # Prepare the "A" parameter
        A = -torch.exp(self.A_log.float())  # [intermediate_size, ssm_state_size]

        # Expand A across the sequence dimension to get discrete_A
        discrete_A = torch.exp(A[None, :, None, :] * discrete_time_step[:, :, :, None])
        # shape => [batch_size, intermediate_size, seq_len, ssm_state_size]
        self.A_discrete = discrete_A.detach().cpu()

        # The input-dependent B
        discrete_B = discrete_time_step[:, :, :, None] * B[:, None, :, :].float()
        # shape => [batch_size, intermediate_size, seq_len, ssm_state_size]
        #print('Hidden state size before ssm: ', hidden_states.shape)
        # Multiply by hidden_states to get the final deltaB_u
        deltaB_u = discrete_B * hidden_states[:, :, :, None].float()
        # shape => [batch_size, intermediate_size, seq_len, ssm_state_size]

        # 3.c. Perform the recurrence: we accumulate state across each time step
        scan_outputs = []
        for i in range(seq_len):
            # Update the ssm_state for the current step
            ssm_state = discrete_A[:, :, i, :] * ssm_state + deltaB_u[:, :, i, :]

            # "scan_output" from the SSM part (before gate or D multiplication)
            scan_output_i = torch.matmul(
                ssm_state.to(dtype),  # [batch_size, intermediate_size, ssm_state_size]
                C[:, i, :].unsqueeze(-1)
            )
            # shape => [batch_size, intermediate_size, 1]
            scan_outputs.append(scan_output_i[:, :, 0])  # remove trailing dim

        # Combine the per-step outputs into one tensor => [batch_size, intermediate_size, seq_len]
        scan_output = torch.stack(scan_outputs, dim=-1)
        #print('before u', scan_output.shape)

        # In your code snippet, there's a line:
        #   scan_output = scan_output + (hidden_states * self.D[None, :, None])
        #
        # We'll store this result BEFORE gate multiplication
        scan_output_no_gate = scan_output + (hidden_states * self.D[None, :, None])
        #print("after u", scan_output_no_gate.shape)

        # Here is the key: store the pre-gate scan_output in a module attribute
        # so we can retrieve it after forward pass. We detach it from the graph
        # to avoid messing with backprop.
        #self.scan_output_no_gate = scan_output_no_gate.detach().cpu()

        # Next, apply the gate:
        #   scan_output = scan_output_no_gate * self.act(gate)
        scan_output = scan_output_no_gate * self.act(gate)

        #self.scan_output_with_gate = scan_output.detach().cpu()

        # If caching is enabled, update ssm_state in the cache
        if cache_params is not None:
            cache_params.update_ssm_state(self.layer_idx, ssm_state)

        # 4. Final linear projection back to [batch_size, seq_len, hidden_size]
        contextualized_states = self.out_proj(scan_output.transpose(1, 2))

        # Done! Return the final outputs as usual.
        return contextualized_states

    # Return our new slow_forward function
    return slow_forward_hooked



In [2]:
# load modified model
model, tokenizer = load_model(quantize=False)
FalconMambaMixer.slow_forward = make_slow_forward_with_hook(
    FalconMambaMixer.slow_forward
)

The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
model

FalconMambaForCausalLM(
  (backbone): FalconMambaModel(
    (embeddings): Embedding(65024, 4096)
    (layers): ModuleList(
      (0-63): 64 x FalconMambaBlock(
        (norm): FalconMambaRMSNorm(4096, eps=1e-05)
        (mixer): FalconMambaMixer(
          (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192)
          (act): SiLU()
          (in_proj): Linear(in_features=4096, out_features=16384, bias=False)
          (x_proj): Linear(in_features=8192, out_features=288, bias=False)
          (dt_proj): Linear(in_features=256, out_features=8192, bias=True)
          (out_proj): Linear(in_features=8192, out_features=4096, bias=False)
        )
      )
    )
    (norm_f): FalconMambaRMSNorm(4096, eps=1e-05)
  )
  (lm_head): Linear(in_features=4096, out_features=65024, bias=False)
)

In [7]:
# Apply your hook
original_slow_forward = FalconMambaMixer.slow_forward
FalconMambaMixer.slow_forward = make_slow_forward_with_hook(original_slow_forward)

# Verification
def verify_hook_applied():
    # Check if the current method is different from the original
    if FalconMambaMixer.slow_forward is not original_slow_forward:
        print("✅ Hook successfully applied to FalconMambaMixer.slow_forward")
        
        # Further verification - check if it's our specific function
        if FalconMambaMixer.slow_forward.__name__ == make_slow_forward_with_hook(original_slow_forward).__name__:
            print("✅ The applied hook matches our expected hook")
        else:
            print("❌ A hook was applied, but it doesn't match our expected hook")
    else:
        print("❌ Hook NOT applied - slow_forward is still the original method")

verify_hook_applied()

# Additional verification: Check a specific instance
for name, module in model.named_modules():
    if isinstance(module, FalconMambaMixer):
        print(f"Layer {module.layer_idx}: Method ID check - {id(module.slow_forward) != id(original_slow_forward)}")

✅ Hook successfully applied to FalconMambaMixer.slow_forward
✅ The applied hook matches our expected hook
Layer 0: Method ID check - True
Layer 1: Method ID check - True
Layer 2: Method ID check - True
Layer 3: Method ID check - True
Layer 4: Method ID check - True
Layer 5: Method ID check - True
Layer 6: Method ID check - True
Layer 7: Method ID check - True
Layer 8: Method ID check - True
Layer 9: Method ID check - True
Layer 10: Method ID check - True
Layer 11: Method ID check - True
Layer 12: Method ID check - True
Layer 13: Method ID check - True
Layer 14: Method ID check - True
Layer 15: Method ID check - True
Layer 16: Method ID check - True
Layer 17: Method ID check - True
Layer 18: Method ID check - True
Layer 19: Method ID check - True
Layer 20: Method ID check - True
Layer 21: Method ID check - True
Layer 22: Method ID check - True
Layer 23: Method ID check - True
Layer 24: Method ID check - True
Layer 25: Method ID check - True
Layer 26: Method ID check - True
Layer 27: Met